# VacationPy
---

## Starter Code to Import Libraries and Load the Weather and Coordinates Data

In [165]:
# Dependencies and Setup
import hvplot.pandas
import pandas as pd
import requests

# Import API key
from api_keys import geoapify_key

In [184]:
# Load the CSV file created in Part 1 into a Pandas DataFrame
city_data_df = pd.read_csv("output_data/cities.csv")

# Display sample data
city_data_df.head(10)

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
0,0,nar'yan-mar,67.6713,53.0870,-7.65,97,97,1.77,RU,1739330251
1,1,west island,-12.1568,96.8225,28.99,84,75,7.72,CC,1739330471
2,2,cockburn town,21.4612,-71.1419,25.33,80,5,9.95,TC,1739330472
3,3,kabalo,-6.0500,26.9167,20.78,97,100,0.69,CD,1739330473
4,4,bethel,41.3712,-73.4140,-0.55,70,100,0.45,US,1739330442
5,5,hermanus,-34.4187,19.2345,19.33,84,0,4.66,ZA,1739330475
6,6,chifeng,42.2683,118.9636,-5.49,32,0,7.32,CN,1739330476
7,7,adamstown,-25.0660,-130.1015,25.76,72,86,0.37,PN,1739330477
8,8,caleta de carquin,-11.0925,-77.6267,23.57,80,100,1.88,PE,1739330478
9,9,borkum,53.5809,6.6915,1.15,97,100,8.72,DE,1739330211


---

### Step 1: Create a map that displays a point for every city in the `city_data_df` DataFrame. The size of the point should be the humidity in each city.

In [187]:
%%capture --no-display

humidity = city_data_df["Humidity"].astype(int)

# Configure the map plot
map_plot = city_data_df.hvplot.points(
    "Lng",
    "Lat",
    geo = True,
    tiles = "OSM",
    frame_width = 800,
    frame_height = 600,
    size = "Humidity",
    color="City",
    alpha=.75
)

# Display the map
map_plot

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Lng,Lat]   (City,Humidity)

### Step 2: Narrow down the `city_data_df` DataFrame to find your ideal weather condition

A max temperature lower than 27 degrees but higher than 21

Wind speed less than 4.5 m/s

Zero cloudiness

In [190]:
# Narrow down cities that fit criteria and drop any results with null values
city_data_df= city_data_df.loc[(city_data_df["Max Temp"] <= 27) & (city_data_df["Max Temp"] > 21)]

#Wind speed less than 4.5 m/s
city_data_df = city_data_df.loc[(city_data_df["Wind Speed"] < 4.5)]

#Zero cloudiness
city_data_df = city_data_df.loc[(city_data_df["Cloudiness"] == 0)]

# Drop results with null values
city_data_df.dropna()

# Display sample data
city_data_df.head()

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
27,27,beaufort west,-32.3567,22.5830,21.15,35,0,2.48,ZA,1739330501
57,57,san rafael,-34.6177,-68.3301,21.22,18,0,2.41,AR,1739330534
80,80,robore,-18.3333,-59.7500,21.72,96,0,1.65,BO,1739330565
82,82,matias romero,16.8833,-95.0333,22.11,85,0,2.22,MX,1739330568
115,115,wailua homesteads,22.0669,-159.3780,24.44,84,0,3.09,US,1739330607


### Step 3: Create a new DataFrame called `hotel_df`.

In [195]:
# Use the Pandas copy function to create DataFrame called hotel_df to store the city, country, coordinates, and humidity

hotel_df = city_data_df[["City", "Country", "Lat", "Lng", "Humidity"]].copy()

# Add an empty column, "Hotel Name," to the DataFrame so you can store the hotel found using the Geoapify API
hotel_df["Hotel Name"] = ""

# Display sample data
hotel_df.head()

,City,Country,Lat,Lng,Humidity,Hotel Name
27,beaufort west,ZA,-32.3567,22.5830,35,
57,san rafael,AR,-34.6177,-68.3301,18,
80,robore,BO,-18.3333,-59.7500,96,
82,matias romero,MX,16.8833,-95.0333,85,
115,wailua homesteads,US,22.0669,-159.3780,84,


### Step 4: For each city, use the Geoapify API to find the first hotel located within 10,000 metres of your coordinates.

In [198]:
# Set parameters to search for a hotel
radius = 10000
params = {
    "categories": "accommodation.hotel",
    "apiKey":geoapify_key
}

# Print a message to follow up the hotel search
print("Hotel search")

# Iterate through the hotel_df DataFrame
for index, row in hotel_df.iterrows():
    # get latitude, longitude from the DataFrame
    lat = row["Lat"]
    lon = row["Lng"]

    # Add the current city's latitude and longitude to the params dictionary
   # Add filter and bias parameters with the current city's latitude and longitude to the params dictionary
    params["filter"] = f"circle:{lon},{lat},{radius}"
    params["bias"] = f"proximity:{lon},{lat}"

    # Set base URL
    base_url = "https://api.geoapify.com/v2/places"

    # Make and API request using the params dictionary
    name_address = requests.get(base_url, params=params)

    # Convert the API response to JSON format
    name_address = name_address.json()

    # Grab the first hotel from the results and store the name in the hotel_df DataFrame
    try:
        hotel_df.loc[index, "Hotel Name"] = name_address["features"][0]["properties"]["name"]
    except (KeyError, IndexError):
        # If no hotel is found, set the hotel name as "No hotel found".
        hotel_df.loc[index, "Hotel Name"] = "No hotel found"

    # Log the search results
    print(f"{hotel_df.loc[index, 'City']} - nearest hotel: {hotel_df.loc[index, 'Hotel Name']}")

# Display sample data
hotel_df

Hotel search
beaufort west - nearest hotel: Matoppo Inn
san rafael - nearest hotel: Hotel Regional
robore - nearest hotel: Lajas
matias romero - nearest hotel: Hotel Gyl Mary
wailua homesteads - nearest hotel: Hilton Garden Inn Kauai Wailua Bay
dwarka - nearest hotel: The Dwarika Hotel
whakatane - nearest hotel: Whakatane Hotel
lihue - nearest hotel: Kauai Palms
kapa'a - nearest hotel: Pono Kai Resort
tura - nearest hotel: Hotel Polo Orchid
brisas de zicatela - nearest hotel: Casa de Olas
chauk - nearest hotel: Royal Chauk


,City,Country,Lat,Lng,Humidity,Hotel Name
27,beaufort west,ZA,-32.3567,22.5830,35,Matoppo Inn
57,san rafael,AR,-34.6177,-68.3301,18,Hotel Regional
80,robore,BO,-18.3333,-59.7500,96,Lajas
82,matias romero,MX,16.8833,-95.0333,85,Hotel Gyl Mary
115,wailua homesteads,US,22.0669,-159.3780,84,Hilton Garden Inn Kauai Wailua Bay
159,dwarka,IN,22.2394,68.9678,78,The Dwarika Hotel
225,whakatane,NZ,-37.9585,176.9854,60,Whakatane Hotel
248,lihue,US,21.9789,-159.3672,73,Kauai Palms
300,kapa'a,US,22.0752,-159.3190,84,Pono Kai Resort
301,tura,IN,25.5198,90.2201,33,Hotel Polo Orchid


### Step 5: Add the hotel name and the country as additional information in the hover message for each city in the map.

In [201]:
%%capture --no-display

# Configure the map plot
map_plot_hotel = hotel_df.hvplot.points(
    "Lng",
    "Lat",
    geo = True,
    tiles = "OSM",
    frame_width = 800,
    frame_height = 600,
    size = "Humidity",
    color="City",
    alpha=.75,
    hover_cols=["Hotel Name", "Country"]
)

# Display the map
map_plot_hotel

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Lng,Lat]   (City,Humidity,Hotel Name,Country)